# D1 · How many trees do you need? Reading OOB curves, test curves and disagreement between forests

**Type**: Interpret · **Level**: ●●○ independent · **Time**: about 50 minutes · **core path ★**

## Objective

Read how the OOB error, the test error and the **disagreement between two forests with different seeds** change as `n_estimators` grows, on two different problems: an "easy" one (the error settles early) and one with hard interactions (the error keeps dropping). Conclude when "a budget of 100 trees" is enough and when it is not.

## Dataset

`easy`: `make_classification(2000, 12 features, 5 informative, flip_y=0.08)`. `interactions`: 2000 points with 10 Gaussian features and `y = (x0·x1 > 0) XOR (x2 > 0.3)` with 5% of swapped labels. 60/40 split; `warm_start` to add trees.

## Educational purpose

Replace the rule "100-500 trees are fine" with reading the curves of the problem: how fast the error settles depends on how hard the signal is, and the **predictions** settle later than the accuracy.

## Task

1. For each problem and for `n` in `[5, 20, 50, 100, 250, 500]` train with `warm_start=True` (increasing `n_estimators`, `oob_score=True`, `random_state=0`) and save `oob_err` and `test_err` (`pd.DataFrame` index `n`, columns `easy` and `interactions`, errors = 1 − accuracy).
2. Save `disagreement` (same shape): for each `n`, the mean over the pairs of **6 independent forests** (`random_state=0..5`) of the fraction of test examples on which the predictions (`predict`) differ.
3. Save `stable_n_error` (dictionary problem → the smallest `n` at which `oob_err` is within **0.01** of the value at 500) and `stable_n_predictions` (the smallest `n` at which `disagreement < 0.05`).
4. Answer: `needs_more_trees` (name of the problem with the largest `stable_n_error`) and `predictions_still_unstable` (True if, in the problem with interactions, the disagreement at 500 trees exceeds 0.05 even though the OOB error is already stable).

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X1, y1 = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
rng = np.random.default_rng(0)
X2 = rng.normal(size=(2000, 10))
y2 = ((X2[:, 0] * X2[:, 1] > 0) ^ (X2[:, 2] > 0.3)).astype(int)
y2 = np.where(rng.random(2000) < 0.05, 1 - y2, y2)
problems = {"easy": (X1, y1), "interactions": (X2, y2)}
splits = {name: train_test_split(X, y, test_size=0.4, random_state=0) for name, (X, y) in problems.items()}
ns = [5, 20, 50, 100, 250, 500]

## Your work

In [ ]:
oob_err = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
test_err = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
for name, (X_tr, X_te, y_tr, y_te) in splits.items():
    f = RandomForestClassifier(warm_start=True, oob_score=True, random_state=0, n_jobs=-1)
    for n in ns:
        ...
pd.concat({"OOB": oob_err, "test": test_err}, axis=1).round(3)

In [ ]:
disagreement = pd.DataFrame(index=ns, columns=list(problems), dtype=float)
disagreement

In [ ]:
stable_n_error = {}
stable_n_predictions = {}
needs_more_trees = None
predictions_still_unstable = None

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='Loop: `f.set_params(n_estimators=n).fit(X_tr, y_tr); oob_err.loc[n, name] = 1 - f.oob_score_; test_err.loc[n, name] = 1 - f.score(X_te, y_te)`. Disagreement: for each `n`, `P = [RandomForestClassifier(n, random_state=s, n_jobs=-1).fit(X_tr, y_tr).predict(X_te) for s in range(6)]` and `np.mean([(P[i] != P[j]).mean() for i in range(6) for j in range(i + 1, 6)])`.'):
    check(oob_err.notna().all().all() and test_err.notna().all().all() and oob_err.loc[5, "easy"] > oob_err.loc[500, "easy"] + 0.03, "complete OOB and test curves; for the easy problem the OOB error drops by over 3 points from 5 to 500 trees", "1 - oob_score_ and 1 - score(X_te, y_te)")
    check(oob_err.loc[500, "interactions"] < oob_err.loc[100, "interactions"] - 0.05 and oob_err.loc[100, "easy"] - oob_err.loc[500, "easy"] < 0.02, "with the interactions the OOB error keeps dropping by over 5 points between 100 and 500 trees; in the easy problem by less than 2", "read the two columns")
    check(disagreement.notna().all().all() and disagreement["easy"].is_monotonic_decreasing and disagreement.loc[5, "easy"] > 0.1 and disagreement.loc[500, "easy"] < 0.05, "the disagreement between forests drops: over 0.10 with 5 trees, under 0.05 with 500 (easy problem)", "mean over the pairs of the 6 forests")
    check(stable_n_error["easy"] <= 100 and stable_n_error["interactions"] >= 250, "the easy problem settles within 100 trees; the interactions need at least 250", "oob_err within 0.01 of the value at 500")
    check(needs_more_trees == "interactions" and predictions_still_unstable is True and disagreement.loc[500, "interactions"] > 0.05 and stable_n_predictions["easy"] <= 100, "the interactions need more trees and at 500 the predictions still disagree on over 5%; the easy problem has stable predictions with 100 trees or fewer", "compare the two dictionaries and the table")
done()

---
The commented solution is in `solutions/D_interpret/D1_how_many_trees_do_you_really_need.ipynb`. Open it after you pass the check, or after you have really tried.